# SFT: Qwen3.5-9B on domain QA + tool calling + general chat
Run cells top to bottom. All settings live in `../../config.yaml`.

In [ ]:
!pip install -q -r ../../requirements/training.txt

## 1. Get the data
Downloads the S3 pool (needs `AWS_*` env vars) and builds `train.jsonl` / `val.jsonl`.

In [ ]:
%run ../../01_data_prep/download_s3.py
%run ../../01_data_prep/build_mixture.py

In [ ]:
import json, pathlib
root = pathlib.Path('../..')
print(json.dumps(json.load(open(root/'data/processed/stats.json')), indent=2))
print(json.loads(open(root/'data/processed/train.jsonl').readline())['text'][:1500])

## 2. Train
Run the checks first (no training): `--memory-check` for peak GPU memory, and in packed mode
`--check-packing` to confirm packed rows stay apart. Training then saves every checkpoint,
its logs and the final adapter to `storage.s3_root/<run>/`. Resume with `--resume`.

In [ ]:
%run ../sft_train.py --memory-check
%run ../sft_train.py --check-packing


In [ ]:
# Full training run (hours). Add --resume to continue after an interruption.
%run ../sft_train.py


## 3. Next
`python 03_merging/merge_model.py` merges the adapter (saved to S3 `merged/`), then
`python 04_evaluation/evaluate.py` compares pre-tuned vs fine-tuned (saved to S3 `eval/`).